# LC #239 — Sliding Window Maximum
**Category:** Deque (Monotonic Queue) | **Difficulty:** Hard | **Day 2**

---

<div style="padding:15px;border-left:8px solid #11998e;background:#e8faf5;border-radius:4px;">
<strong>The Problem:</strong> Given array <code>nums</code> and integer <code>k</code>, return an array
of the maximum value in each sliding window of size k.
</div>

**Examples:**
```
nums = [1,3,-1,-3,5,3,6,7], k=3 → [3,3,5,5,6,7]
nums = [1],                  k=1 → [1]
nums = [1,-1],               k=1 → [1,-1]
```

**Core Insight:** Maintain a **monotonic decreasing deque** of indices. The front always holds the index of the current window's maximum. Remove from the front when that index falls out of the window. Remove from the back when a larger element enters (those smaller elements can never become the max).

## Mental Models

**1. The Monotonic Deque as a "Useful Candidates" List**
Only elements that could become the window max are worth keeping. If element `b` is smaller than and to the left of element `a`, then `b` can never be the max while `a` is in the window. Remove `b`.

**2. Deque Front = Current Max**
The deque is maintained in decreasing order of values (indices stored). The front index always points to the current window's maximum.

**3. Two Operations**
- Left cleanup: Pop front if its index is outside the window (`dq[0] < i - k + 1`)
- Right cleanup: Pop back while `nums[back] < nums[current]` (smaller elements are useless)

**4. Amortized O(n)**
Each element is added and removed from the deque at most once — so despite inner while loops, total work is O(n).

In [ ]:
# Brute Force — O(n*k) time
def maxSlidingWindow_brute(nums, k):
    result = []
    for i in range(len(nums) - k + 1):
        result.append(max(nums[i:i+k]))
    return result

print(maxSlidingWindow_brute([1,3,-1,-3,5,3,6,7], 3))   # [3,3,5,5,6,7]
print(maxSlidingWindow_brute([1], 1))                    # [1]

## Trace: `[1,3,-1,-3,5,3,6,7]`, k=3

| i | num | Front cleanup | Back cleanup | dq (indices) | Window | Result |
|---|-----|--------------|-------------|-------------|--------|--------|
| 0 | 1 | — | — | [0] | — | — |
| 1 | 3 | — | pop 0 (1<3) | [1] | — | — |
| 2 | -1 | — | -1<3, keep | [1,2] | full | **3** |
| 3 | -3 | — | -3<-1, keep | [1,2,3] | drop 0 → [1,2,3] | **3** |
| 4 | 5 | pop 1 (out) | pop 3,2 | [4] | | **5** |
| 5 | 3 | — | 3<5, keep | [4,5] | | **5** |
| 6 | 6 | — | pop 5,4 | [6] | | **6** |
| 7 | 7 | — | pop 6 | [7] | | **7** |

In [ ]:
# Optimal — O(n) time, O(k) space
from collections import deque

def maxSlidingWindow(nums: list[int], k: int) -> list[int]:
    dq = deque()   # stores indices, values are decreasing
    result = []

    for i, num in enumerate(nums):
        # Remove indices outside the current window
        while dq and dq[0] < i - k + 1:
            dq.popleft()

        # Remove indices whose values are smaller than current (they're useless)
        while dq and nums[dq[-1]] < num:
            dq.pop()

        dq.append(i)

        # Window is full — record the max (front of deque)
        if i >= k - 1:
            result.append(nums[dq[0]])

    return result

# Test
print(maxSlidingWindow([1,3,-1,-3,5,3,6,7], 3))   # [3,3,5,5,6,7]
print(maxSlidingWindow([1], 1))                    # [1]
print(maxSlidingWindow([9,11], 2))                 # [11]
print(maxSlidingWindow([4,-2], 2))                 # [4]

## Complexity Analysis

| Approach | Time | Space |
|----------|------|-------|
| Brute Force | O(n×k) | O(1) |
| **Monotonic Deque** | **O(n)** | **O(k)** |

**Why O(n) despite inner loops?** Each element enters and exits the deque at most once. Amortized, the total number of push+pop operations across all iterations is ≤ 2n.

**Why O(k) space?** The deque holds at most k indices at any time (one window's worth).

## Interview Q&A

**Q: Why store indices instead of values in the deque?**
A: We need to check if the front element has fallen outside the window. Comparing indices to `i - k + 1` requires the index, not the value.

**Q: What makes this "monotonic"?**
A: The values corresponding to indices in the deque are always in decreasing order. We maintain this monotonic property by removing smaller elements from the back on every insertion.

**Q: Real-world use case?**
A: Rolling maximum of a time series — "what was the peak CPU in the last 5 minutes?" With streaming data arriving per-second, this runs in O(1) per new reading rather than O(k) per reading.

**Q: How would you adapt this for rolling minimum?**
A: Flip the comparison: remove from back when `nums[dq[-1]] > num` (maintain increasing order). Front will always be the minimum.

## The Citi Angle

**Direct application: rolling peak CPU detection.**

```python
# Rolling maximum CPU over a k-window — O(n) with monotonic deque
from collections import deque

def rolling_peak(readings: list[float], window: int) -> list[float]:
    """Returns peak CPU for each window position. O(n) time."""
    dq = deque()
    peaks = []
    for i, cpu in enumerate(readings):
        while dq and dq[0] < i - window + 1:
            dq.popleft()
        while dq and readings[dq[-1]] < cpu:
            dq.pop()
        dq.append(i)
        if i >= window - 1:
            peaks.append(readings[dq[0]])
    return peaks

# 6 hourly CPU readings, 3-hour rolling peak
readings = [72.1, 68.5, 81.3, 75.0, 90.2, 88.7, 65.4]
print("Rolling 3-hour peak CPU:", rolling_peak(readings, 3))
```

**Interview tie-in:** "At Citi, identifying the peak CPU in a rolling window was a core capacity alert. The monotonic deque gives O(n) — critical for streaming telemetry where n grows continuously."

## Summary

| | Value |
|--|--|
| **Pattern** | Monotonic Deque |
| **Time** | O(n) amortized |
| **Space** | O(k) |
| **Deque invariant** | Values are decreasing (indices stored; front = current max) |
| **Say in interview** | "Monotonic deque: remove expired front, remove smaller elements from back. Front is always the window max. Each element enters and exits at most once — O(n) amortized." |